# CNN inverse training on Colab T4

Workflow: clone repo from GitHub → install as package (pulls all deps from pyproject.toml) → generate dataset → train CNN → evaluate.

To iterate on hyperparameters: edit `notebooks/train_cnn_inverse.py` locally → `git push` → in this notebook run the `git pull` cell → re-run the training cell.

In [ ]:
# 1) Clone repo, install as package (with all deps), verify GPU
import os, sys, subprocess

REPO_DIR = "/content/water_v2"
REPO_URL = "https://github.com/alexhrubin/water_v2.git"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", "python-rewrite", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"{REPO_DIR} already exists; run the `git pull` cell below to update.")

os.chdir(REPO_DIR)

# Install as editable package — pulls in jax, jaxopt, equinox, optax, scipy, matplotlib, etc.
# from pyproject.toml. Quiet flag keeps output short; check=True surfaces install errors.
subprocess.run(["pip", "install", "-q", "-e", REPO_DIR], check=True)

import jax
print(f"JAX backend: {jax.default_backend()}  devices: {jax.devices()}")
from wavetank import Tank, build_propagator   # noqa: F401
print("wavetank imports OK")

In [ ]:
# Pull latest changes after editing scripts locally and `git push`-ing
!cd /content/water_v2 && git pull

In [ ]:
# 2) Generate 1M-sample dataset on GPU (~1 min on T4)
# Writes to data/naive_inverse/dataset.npz (~17 GB) inside Colab session storage.
!python -u notebooks/gen_naive_inverse_data.py

In [ ]:
# 3) Train CNN (~5-10 min on T4)
!python -u notebooks/train_cnn_inverse.py

In [ ]:
# 4) OOD eval
!python -u notebooks/eval_cnn_inverse.py

In [ ]:
from IPython.display import Image, display
display(Image("data/naive_inverse/eval_cnn_ood.png"))